# Mission EdStats — qualité des données

Mark veut savoir si les données EdStats peuvent guider l'expansion internationale d'academy : quels pays cibler, et comment ce potentiel évolue.

Avant de chercher des pays, on vérifie les 5 fichiers, un par un : ce qu'est une ligne, les doublons, les valeurs manquantes, les colonnes à garder.

## Mise en place

### 1. Imports et dossier des données

`RESOURCES_DIR` pointe vers le dossier `resources/` des 5 CSV. Le chemin dépend de l'endroit d'où le notebook est lancé : on teste deux candidats, on garde le premier qui existe.

In [1]:
import pandas as pd
from pathlib import Path

candidates = [
    Path.cwd() / "notebooks" / "resources",  # lancé depuis la racine du projet
    Path.cwd() / "resources",                 # lancé depuis le dossier notebooks/
]

RESOURCES_DIR = None
for candidate in candidates:
    if candidate.exists():
        RESOURCES_DIR = candidate.resolve()  # chemin absolu
        break

if RESOURCES_DIR is None:
    raise FileNotFoundError("Dossier resources/ introuvable")

RESOURCES_DIR

PosixPath('/Users/emmanuel/Formation Data Engineer/Projets/P2 - Analyse données systèmes éducatifs/notebooks/resources')

### 2. La fonction de chargement brut

`load_csv_analysis()` lit un CSV tel quel, sans nettoyage. Deux réglages : `utf-8` pour les accents, et les colonnes « Unnamed » écartées — chaque ligne du CSV finit par une virgule, ça crée une colonne vide. Un artefact du fichier, pas une donnée.

On ne remplace pas les valeurs manquantes : on veut justement les compter.

In [2]:
def load_csv_analysis(name):
    """Lit le CSV `name` depuis RESOURCES_DIR, tel quel (brut), pour l'analyse."""
    return pd.read_csv(
        RESOURCES_DIR / name,
        encoding="utf-8",
        usecols=lambda column: "Unnamed" not in column,
    )

### 3. Chargement des 5 CSV

Les 5 fichiers sont chargés dans `dfs` : un DataFrame par fichier. `shape` donne (lignes, colonnes). `EdStatsData.csv` fait ~326 Mo, cette cellule peut prendre une minute.

In [3]:
FICHIERS = [
    "EdStatsCountry.csv",        # référentiel des pays et agrégats
    "EdStatsSeries.csv",         # dictionnaire des indicateurs
    "EdStatsData.csv",           # mesures par pays, indicateur et année
    "EdStatsCountry-Series.csv", # sources par couple pays × indicateur
    "EdStatsFootNote.csv",       # annotations par pays, indicateur et année
]

dfs = {}
for name in FICHIERS:
    dfs[name] = load_csv_analysis(name)
    print(f"{name} : {dfs[name].shape[0]} lignes, {dfs[name].shape[1]} colonnes")

EdStatsCountry.csv : 241 lignes, 31 colonnes
EdStatsSeries.csv : 3665 lignes, 20 colonnes


EdStatsData.csv : 886930 lignes, 69 colonnes
EdStatsCountry-Series.csv : 613 lignes, 3 colonnes
EdStatsFootNote.csv : 643638 lignes, 4 colonnes


### 4. Taille de chaque fichier

In [4]:
recap = pd.DataFrame(
    [(name, df.shape[0], df.shape[1]) for name, df in dfs.items()],
    columns=["Fichier", "Lignes", "Colonnes"],
)
recap

,Fichier,Lignes,Colonnes
0,EdStatsCountry.csv,241,31
1,EdStatsSeries.csv,3665,20
2,EdStatsData.csv,886930,69
3,EdStatsCountry-Series.csv,613,3
4,EdStatsFootNote.csv,643638,4


Aperçu brut du référentiel des pays :

In [5]:
dfs["EdStatsCountry.csv"].head(3)

,Country Code,Short Name,Table Name,Long Name,2-alpha code,Currency Unit,Special Notes,Region,Income Group,WB-2 code,...,Government Accounting concept,IMF data dissemination standard,Latest population census,Latest household survey,Source of most recent Income and expenditure data,Vital registration complete,Latest agricultural census,Latest industrial data,Latest trade data,Latest water withdrawal data
0,ABW,Aruba,Aruba,Aruba,AW,Aruban florin,SNA data for 2000-2011 are updated from offici...,Latin America & Caribbean,High income: nonOECD,AW,...,NaN,NaN,2010,NaN,NaN,Yes,NaN,NaN,2012.0,NaN
1,AFG,Afghanistan,Afghanistan,Islamic State of Afghanistan,AF,Afghan afghani,Fiscal year end: March 20; reporting period fo...,South Asia,Low income,AF,...,Consolidated central government,General Data Dissemination System (GDDS),1979,"Multiple Indicator Cluster Survey (MICS), 2010/11","Integrated household survey (IHS), 2008",NaN,2013/14,NaN,2012.0,2000
2,AGO,Angola,Angola,People's Republic of Angola,AO,Angolan kwanza,"April 2013 database update: Based on IMF data,...",Sub-Saharan Africa,Upper middle income,AO,...,Budgetary central government,General Data Dissemination System (GDDS),1970,"Malaria Indicator Survey (MIS), 2011","Integrated household survey (IHS), 2008",NaN,2015,NaN,NaN,2005


### Bilan de la mise en place

Les 5 CSV sont chargés bruts dans `dfs`, réservé à l'affichage : il ne sera ni modifié ni calculé. Chaque section qui suit rechargera son propre CSV brut pour calculer.

Prochaine étape : les fichiers un par un, en commençant par `EdStatsCountry`.

## EdStatsCountry.csv — référentiel des pays

*Une ligne = un pays ou un agrégat (ex. « Arab World »).*

### 1. Une ligne = un pays ou un agrégat

Chaque ligne a son `Country Code` (ex. `FRA`). C'est la clé qui joindra les fichiers : elle ne doit jamais être en double.

In [6]:
# DataFrame dédié au calcul pour cette section : on recharge le CSV brut,
# séparément du `dfs` réservé à l'affichage
df = load_csv_analysis("EdStatsCountry.csv")

cle = "Country Code"  # clé métier (PK) d'après le dictionnaire de data_overview.ipynb
print(f"{len(df)} lignes, {df[cle].nunique()} valeurs distinctes de '{cle}'")
df[cle].is_unique  # True = chaque valeur de la clé n'apparaît que sur une seule ligne

241 lignes, 241 valeurs distinctes de 'Country Code'


True

241 lignes, 241 codes différents : la clé est unique, la jointure ne créera pas de doublons.

### 2. Taille du fichier

In [7]:
df.shape

(241, 31)

241 lignes × 31 colonnes : 31 renseignements par pays ou agrégat.

### 3. Des doublons ?

`duplicated()` repère les lignes identiques à une autre, `drop_duplicates()` les supprimerait. Un doublon ici, ce serait le même pays deux fois.

In [8]:
nb_doublons = df.duplicated().sum()
print(f"{nb_doublons} ligne(s) en double (toutes colonnes confondues)")
if nb_doublons > 0:
    df = df.drop_duplicates()
    print(f"drop_duplicates() appliqué : il reste {df.shape[0]} lignes")
else:
    print("aucun doublon : drop_duplicates() inutile, le DataFrame est inchangé")

0 ligne(s) en double (toutes colonnes confondues)
aucun doublon : drop_duplicates() inutile, le DataFrame est inchangé


0 doublon : aucun pays en double, aucune information répétée sur deux lignes.

### 4. Valeurs manquantes et taux de remplissage

`isnull()` repère les cellules vides. On compte, colonne par colonne, ce qui manque : c'est le taux de remplissage. On ne remplace rien — on veut justement voir les trous.

In [9]:
remplissage = pd.DataFrame({
    "nb_manquants": df.isnull().sum(),
    "proportion_manquante_%": (df.isnull().mean() * 100).round(2),
    "taux_remplissage_%": ((1 - df.isnull().mean()) * 100).round(2),
}).sort_values("taux_remplissage_%", ascending=False)  # les mieux remplies en premier

n_completes = (remplissage["taux_remplissage_%"] == 100).sum()  # 100 % = aucune valeur manquante
n_vides = (remplissage["taux_remplissage_%"] == 0).sum()        # 0 % = colonne entièrement vide
print(f"{n_completes} colonne(s) à 100 % de remplissage, {n_vides} colonne(s) vide(s) à 0 %")

remplissage

4 colonne(s) à 100 % de remplissage, 0 colonne(s) vide(s) à 0 %


,nb_manquants,proportion_manquante_%,taux_remplissage_%
Country Code,0,0.00,100.00
Table Name,0,0.00,100.00
Long Name,0,0.00,100.00
Short Name,0,0.00,100.00
WB-2 code,1,0.41,99.59
2-alpha code,3,1.24,98.76
System of National Accounts,26,10.79,89.21
Currency Unit,26,10.79,89.21
Region,27,11.20,88.80
Income Group,27,11.20,88.80


Les 4 colonnes d'identification sont complètes, aucune colonne n'est vide. Les moins remplies descendent à 13,3 % (`National accounts reference year`). `Region`, `Income Group` et `Currency Unit` ne sont remplies qu'à 88,8 % — on verra au point 7 qui manque.

Ce tableau dit sur quoi s'appuyer pour classer les pays, et sur quoi ne pas s'appuyer.

### 5. Colonnes inutilisables à supprimer

Une colonne n'apporte rien si elle est vide à 100 %, constante (une seule valeur : rien pour distinguer les pays) ou redondante avec une autre. Une colonne partiellement vide mais utile (ex. `Region`) reste.

In [10]:
# 1) colonnes vides à 100 % de NaN
vides = df.columns[df.isnull().all()].tolist()
print("colonnes vides à 100 % de NaN :", vides)

# 2) colonnes constantes : une seule valeur renseignée -> aucune information discriminante
constantes = [c for c in df.columns if df[c].nunique() == 1]
print("colonnes constantes (1 seule valeur renseignée) :", constantes)

# 3) colonnes redondantes : "WB-2 code" (code interne BM à 2 lettres) reprend "2-alpha code" (code ISO)
sim = (df["2-alpha code"].fillna("<NA>") == df["WB-2 code"].fillna("<NA>")).mean()
print(f"'2-alpha code' == 'WB-2 code' dans {sim:.0%} des lignes")
redondantes = ["WB-2 code"]

a_supprimer = vides + constantes + redondantes
print("colonnes supprimées :", a_supprimer)
df = df.drop(columns=a_supprimer)
df.shape

colonnes vides à 100 % de NaN : []
colonnes constantes (1 seule valeur renseignée) : ['Balance of Payments Manual in use']
'2-alpha code' == 'WB-2 code' dans 97% des lignes
colonnes supprimées : ['Balance of Payments Manual in use', 'WB-2 code']


(241, 29)

2 colonnes supprimées (31 → 29) : `Balance of Payments Manual in use` (constante) et `WB-2 code` (identique à `2-alpha code` dans 97 % des lignes). Aucune colonne vide à 100 %.

### 6. Statistiques des colonnes numériques

`describe()` résume chaque colonne numérique : nombre de valeurs, moyenne, min, max…

In [11]:
df.select_dtypes(include="number").describe()

,National accounts reference year,Latest industrial data,Latest trade data
count,32.00000,107.000000,185.000000
mean,2001.53125,2008.102804,2010.994595
std,5.24856,2.616834,2.569675
min,1987.00000,2000.000000,1995.000000
25%,1996.75000,2007.500000,2011.000000
50%,2002.00000,2009.000000,2012.000000
75%,2005.00000,2010.000000,2012.000000
max,2012.00000,2010.000000,2012.000000


3 colonnes numériques seulement, toutes des années de dernière donnée macro : rien à en conclure à ce stade. Les autres colonnes d'année sont restées en texte (ex. `2002/03`).

### 7. Les colonnes catégorielles

Les catégories sont les colonnes où les valeurs se répètent (`Region`, `Income Group`…) : c'est elles qui permettront de regrouper et filtrer les pays. On mesure d'abord combien de valeurs distinctes contient chaque colonne texte.

In [12]:
cols_texte = df.select_dtypes(exclude="number").columns

unicite = pd.DataFrame({
    "nb_valeurs_distinctes": [df[c].nunique() for c in cols_texte],
    "part_des_lignes_%": [round(df[c].nunique() / len(df) * 100, 1) for c in cols_texte],
}, index=cols_texte).sort_values("nb_valeurs_distinctes")
unicite

,nb_valeurs_distinctes,part_des_lignes_%
Other groups,2,0.8
Vital registration complete,2,0.8
IMF data dissemination standard,2,0.8
Government Accounting concept,2,0.8
System of trade,2,0.8
SNA price valuation,2,0.8
External debt Reporting status,3,1.2
PPP survey year,3,1.2
System of National Accounts,3,1.2
Lending category,3,1.2


Deux groupes nets, rien entre eux : les colonnes à 2-7 valeurs (les vraies catégories) et celles à 20 et plus (texte ou année, presque une valeur par ligne). Le seuil de 10 tombe dans le trou : n'importe quelle valeur entre 8 et 19 ferait le même tri.

Reste à regarder ce que contiennent ces catégories.

In [13]:
SEUIL = 10  # n'importe quelle valeur entre 8 et 19 donne le même découpage (trou du tableau d'unicité)
cats = [col for col in cols_texte if df[col].nunique() <= SEUIL]

lignes = []
for col in cats:
    for valeur, effectif in df[col].value_counts().items():  # les valeurs existantes, hors cellules vides
        lignes.append({"colonne": col, "valeur": valeur, "effectif": effectif})
inventaire = pd.DataFrame(lignes)
inventaire

,colonne,valeur,effectif
0,Region,Europe & Central Asia,57
1,Region,Sub-Saharan Africa,48
2,Region,Latin America & Caribbean,41
3,Region,East Asia & Pacific,36
4,Region,Middle East & North Africa,21
5,Region,South Asia,8
6,Region,North America,3
7,Income Group,Upper middle income,55
8,Income Group,Lower middle income,50
9,Income Group,High income: nonOECD,44


In [14]:
sans_region = df.loc[df["Region"].isna()]
# Gibraltar et Nauru repérés à la lecture des noms : aucun critère du fichier ne sépare pays et agrégats
pays_sans_region = sans_region.loc[sans_region["Country Code"].isin(["GIB", "NRU"]), ["Country Code", "Short Name"]]
print(f"{len(sans_region)} lignes sans Region : {len(sans_region) - len(pays_sans_region)} agrégats et {len(pays_sans_region)} vrais pays.")
pays_sans_region

27 lignes sans Region : 25 agrégats et 2 vrais pays.


,Country Code,Short Name
78,GIB,Gibraltar
166,NRU,Nauru


12 colonnes catégorielles, de 2 à 7 valeurs. `Region` et `Income Group` classent les 214 pays ayant une région ; les 27 lignes sans région sont des agrégats (World, EU…) et 2 petits pays non classés (Gibraltar, Nauru). Pour chercher des pays-cibles, on gardera les lignes avec une région.

### Bilan EdStatsCountry

`Country Code` est la clé de jointure avec les autres fichiers, et elle est validée : 241 lignes, 241 codes uniques, aucun doublon — aucun pays n'apparaît deux fois.

Chaque ligne décrit un pays ou un agrégat sur 31 colonnes. Le taux de remplissage montre sur quoi s'appuyer et sur quoi ne pas s'appuyer. Deux colonnes inutilisables ont été supprimées (une constante, une redondante) ; les colonnes numériques ne disent que des années de dernière donnée — rien à en conclure à ce stade.

L'essentiel pour la suite : les catégories (`Region`, `Income Group`, `Lending category`…). Elles permettront de regrouper et filtrer les pays quand on cherchera des cibles pour academy.

Prochaine section : `EdStatsSeries.csv`.

## EdStatsSeries.csv — dictionnaire des indicateurs

*Une ligne = un indicateur (ex. `SE.SEC.ENRR`, le taux de scolarisation dans le secondaire).*

### 1. Une ligne = un indicateur

Chaque ligne porte son `Series Code` (ex. `SE.SEC.ENRR`). C'est la clé qui joindra les fichiers : dans `EdStatsData.csv`, la colonne `Indicator Code` pointe ici. Elle ne doit jamais être en double.

In [15]:
# DataFrame dédié au calcul pour cette section : on recharge le CSV brut,
# séparément du `dfs` réservé à l'affichage
df = load_csv_analysis("EdStatsSeries.csv")

cle = "Series Code"  # clé métier (PK) d'après le dictionnaire de data_overview.ipynb
print(f"{len(df)} lignes, {df[cle].nunique()} valeurs distinctes de '{cle}'")
df[cle].is_unique  # True = chaque valeur de la clé n'apparaît que sur une seule ligne

3665 lignes, 3665 valeurs distinctes de 'Series Code'


True

3665 lignes, 3665 codes différents : la clé est unique, la jointure ne créera pas de doublons.

### 2. Taille du fichier

In [16]:
df.shape

(3665, 20)

3665 lignes × 20 colonnes : 20 renseignements par indicateur.

### 3. Des doublons ?

`duplicated()` repère les lignes identiques à une autre, `drop_duplicates()` les supprimerait. Un doublon ici, ce serait le même indicateur deux fois.

In [17]:
nb_doublons = df.duplicated().sum()
print(f"{nb_doublons} ligne(s) en double (toutes colonnes confondues)")
if nb_doublons > 0:
    df = df.drop_duplicates()
    print(f"drop_duplicates() appliqué : il reste {df.shape[0]} lignes")
else:
    print("aucun doublon : drop_duplicates() inutile, le DataFrame est inchangé")

0 ligne(s) en double (toutes colonnes confondues)
aucun doublon : drop_duplicates() inutile, le DataFrame est inchangé


0 doublon : aucun indicateur en double, aucune information répétée sur deux lignes.

### 4. Valeurs manquantes et taux de remplissage

`isnull()` repère les cellules vides. On compte, colonne par colonne, ce qui manque : c'est le taux de remplissage. On ne remplace rien — on veut justement voir les trous.

In [18]:
remplissage = pd.DataFrame({
    "nb_manquants": df.isnull().sum(),
    "proportion_manquante_%": (df.isnull().mean() * 100).round(2),
    "taux_remplissage_%": ((1 - df.isnull().mean()) * 100).round(2),
}).sort_values("taux_remplissage_%", ascending=False)  # les mieux remplies en premier

n_completes = (remplissage["taux_remplissage_%"] == 100).sum()  # 100 % = aucune valeur manquante
n_vides = (remplissage["taux_remplissage_%"] == 0).sum()        # 0 % = colonne entièrement vide
print(f"{n_completes} colonne(s) à 100 % de remplissage, {n_vides} colonne(s) vide(s) à 0 %")

remplissage

5 colonne(s) à 100 % de remplissage, 5 colonne(s) vide(s) à 0 %


,nb_manquants,proportion_manquante_%,taux_remplissage_%
Series Code,0,0.00,100.00
Source,0,0.00,100.00
Indicator Name,0,0.00,100.00
Long definition,0,0.00,100.00
Topic,0,0.00,100.00
Short definition,1509,41.17,58.83
Other notes,3113,84.94,15.06
Base Period,3351,91.43,8.57
Related source links,3450,94.13,5.87
Periodicity,3566,97.30,2.70


5 colonnes complètes : l'identité de l'indicateur (`Series Code`, `Indicator Name`, `Topic`), sa définition longue et sa source. 5 colonnes entièrement vides. Entre les deux, tout est clairsemé : `Short definition` plafonne à 58,8 %, plus rien ne dépasse 16 %. Un dictionnaire tient sur l'essentiel — le reste est du contexte optionnel.

### 5. Colonnes inutilisables à supprimer

Une colonne n'apporte rien si elle est vide à 100 % ou constante (une seule valeur : rien pour distinguer deux indicateurs). Une colonne partiellement vide mais utile (ex. `Short definition`) reste.

In [19]:
# 1) colonnes vides à 100 % de NaN
vides = df.columns[df.isnull().all()].tolist()
print("colonnes vides à 100 % de NaN :", vides)

# 2) colonnes constantes : une seule valeur renseignée -> aucune information discriminante
constantes = [c for c in df.columns if df[c].nunique() == 1]
print("colonnes constantes (1 seule valeur renseignée) :", constantes)

a_supprimer = vides + constantes
print("colonnes supprimées :", a_supprimer)
df = df.drop(columns=a_supprimer)
df.shape

colonnes vides à 100 % de NaN : ['Unit of measure', 'Notes from original source', 'Other web links', 'Related indicators', 'License Type']
colonnes constantes (1 seule valeur renseignée) : ['Periodicity', 'Development relevance', 'Related source links']
colonnes supprimées : ['Unit of measure', 'Notes from original source', 'Other web links', 'Related indicators', 'License Type', 'Periodicity', 'Development relevance', 'Related source links']


(3665, 12)

8 colonnes supprimées (20 → 12) : 5 entièrement vides (`Unit of measure`, `License Type`…) et 3 constantes — `Periodicity` ne dit que « Annual », `Related source links` un seul lien, `Development relevance` un seul texte. Aucune ne peut distinguer deux indicateurs.

### 6. Statistiques des colonnes numériques

`describe()` résume chaque colonne numérique : nombre de valeurs, moyenne, min, max…

In [20]:
numeriques = df.select_dtypes(include="number").columns
print(f"{len(numeriques)} colonne(s) numérique(s) après le nettoyage du point 5")

# d'où pouvaient venir des colonnes numériques ? on regarde le CSV brut, avant suppression
brut = load_csv_analysis("EdStatsSeries.csv")
print("colonnes numériques du CSV brut :", list(brut.select_dtypes(include="number").columns))

0 colonne(s) numérique(s) après le nettoyage du point 5
colonnes numériques du CSV brut : ['Unit of measure', 'Notes from original source', 'Other web links', 'Related indicators', 'License Type']


0 colonne numérique après nettoyage. Celles du CSV brut n'étaient numériques qu'en apparence : des colonnes entièrement vides, que pandas code en float par défaut — exactement les 5 colonnes vides supprimées au point 5. Un dictionnaire, c'est du texte : rien à décrire.

### 7. Les colonnes catégorielles

Les catégories sont les colonnes où les valeurs se répètent (`Topic`, `Source`…) : c'est elles qui permettront de filtrer les indicateurs. On mesure d'abord combien de valeurs distinctes contient chaque colonne texte.

In [21]:
cols_texte = df.select_dtypes(exclude="number").columns

unicite = pd.DataFrame({
    "nb_valeurs_distinctes": [df[c].nunique() for c in cols_texte],
    "part_des_lignes_%": [round(df[c].nunique() / len(df) * 100, 1) for c in cols_texte],
}, index=cols_texte).sort_values("nb_valeurs_distinctes")
unicite

,nb_valeurs_distinctes,part_des_lignes_%
Statistical concept and methodology,2,0.1
Aggregation method,3,0.1
Base Period,4,0.1
General comments,8,0.2
Limitations and exceptions,9,0.2
Other notes,14,0.4
Source,31,0.8
Topic,37,1.0
Short definition,1169,31.9
Long definition,2060,56.2


Deux groupes nets : les colonnes quasi uniques — code, nom, définitions, presque une valeur par ligne — et les vraies catégories, de 2 à 37 valeurs. Entre 37 (`Topic`) et 1169 (`Short definition`), le trou est immense : n'importe quel seuil fait le même tri.

Sur les 8 catégories, seules `Topic` et `Source` sont complètes (point 4) — les 6 autres vivent sous les 16 % de remplissage.

In [22]:
df["Topic"].value_counts().head(10)

Topic
Learning Outcomes                      1046
Attainment                              733
Education Equality                      426
Secondary                               256
Primary                                 248
Population                              213
Tertiary                                158
Teachers                                137
Expenditures                             93
Engaging the Private Sector (SABER)      51
Name: count, dtype: int64

37 thèmes, dominés par l'éducation : `Learning Outcomes` (résultats d'apprentissage) et `Attainment` (niveaux atteints) couvrent à eux deux près de la moitié du dictionnaire. `Topic` est la colonne qui servira à filtrer les indicateurs quand on cherchera des cibles pour academy.

In [23]:
df["Source"].value_counts().head(5)

Source
UNESCO Institute for Statistics                                                                         1269
Early Grade Reading Assessment (EGRA): https://www.eddataglobal.org/reading/                             403
Robert J. Barro and Jong-Wha Lee: http://www.barrolee.com/                                               360
Wittgenstein Centre for Demography and Global Human Capital: http://www.oeaw.ac.at/vid/dataexplorer/     308
Systems Approach for Better Education Results (SABER), World Bank                                        215
Name: count, dtype: int64

31 producteurs, dominés par l'UNESCO Institute for Statistics (1269 indicateurs sur 3665). Les autres catégories (`Base Period`, `Aggregation method`…) sont déjà visibles au point 4 : trop creuses pour servir.

### Bilan EdStatsSeries

`Series Code` est la clé de jointure avec `EdStatsData.csv` — où elle s'appelle `Indicator Code` — et elle est validée : 3665 lignes, 3665 codes uniques, aucun doublon.

Chaque ligne décrit un indicateur sur 20 colonnes ; 8 ne servaient à rien (5 vides, 3 constantes) et ont été supprimées. Aucune colonne numérique ne reste — un dictionnaire, c'est du texte.

L'essentiel pour la suite : `Topic` et ses 37 thèmes, dominés par l'éducation. C'est lui qui permettra de cibler les indicateurs utiles au moment d'analyser `EdStatsData.csv`.

Prochaine section : `EdStatsData.csv`.